In [1]:

# CRACK SEGMENTATION (Frames 1-15)

import cv2
import numpy as np
import os
 
input_folder = "road_member2_gaussian_contrast"
 
# Separate folder for each stage
folders_cell1 = {
    "input":         "segmentation_results/1_input_frames",
    "gray":          "segmentation_results/2_gray_frames",
    "edges":         "segmentation_results/3_edge_frames",
    "morph":         "segmentation_results/4_morph_frames",
    "contours":      "segmentation_results/5_contour_frames",
}
 
# Create all folders
for folder in folders_cell1.values():
    os.makedirs(folder, exist_ok=True)
 
for i in range(1, 16):
 
    # Load enhanced frame
    img_path = os.path.join(input_folder, f"frame_{i}.jpg")
    image = cv2.imread(img_path)
 
    if image is None:
        print(f"Frame {i} not found, skipping...")
        continue
 
    # Save input
    cv2.imwrite(os.path.join(folders_cell1["input"], f"frame_{i}.jpg"), image)
 
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    cv2.imwrite(os.path.join(folders_cell1["gray"], f"frame_{i}.jpg"), gray)
 
    # Edge detection for crack segmentation
    edges = cv2.Canny(gray, 50, 150)
    cv2.imwrite(os.path.join(folders_cell1["edges"], f"frame_{i}.jpg"), edges)
 
    # Morphological operation to connect crack segments
    kernel = np.ones((3, 3), np.uint8)
    morph = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
    cv2.imwrite(os.path.join(folders_cell1["morph"], f"frame_{i}.jpg"), morph)
 
    # Find contours and draw bounding boxes
    contours, _ = cv2.findContours(morph, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    output = image.copy()
 
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 50:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(output, (x, y), (x+w, y+h), (0, 255, 0), 2)
            cv2.putText(output, "Crack", (x, y-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
 
    cv2.imwrite(os.path.join(folders_cell1["contours"], f"frame_{i}.jpg"), output)
 
    print(f"Cell1 - Frame {i} processed and saved.")
 
print("\nCell 1 completed!")
print("Folder structure:")
for name, path in folders_cell1.items():
    print(f"  {path}/")

Cell1 - Frame 1 processed and saved.
Cell1 - Frame 2 processed and saved.
Cell1 - Frame 3 processed and saved.
Cell1 - Frame 4 processed and saved.
Cell1 - Frame 5 processed and saved.
Cell1 - Frame 6 processed and saved.
Cell1 - Frame 7 processed and saved.
Cell1 - Frame 8 processed and saved.
Cell1 - Frame 9 processed and saved.
Cell1 - Frame 10 processed and saved.
Cell1 - Frame 11 processed and saved.
Cell1 - Frame 12 processed and saved.
Cell1 - Frame 13 processed and saved.
Cell1 - Frame 14 processed and saved.
Cell1 - Frame 15 processed and saved.

Cell 1 completed!
Folder structure:
  segmentation_results/1_input_frames/
  segmentation_results/2_gray_frames/
  segmentation_results/3_edge_frames/
  segmentation_results/4_morph_frames/
  segmentation_results/5_contour_frames/


In [2]:
# ALLIGATOR SEGMENTATION (Frames 15-30)
import cv2
import numpy as np
import os

# ====== INPUT FOLDER ======
input_folder = "road_member3_sharpened"

# ====== OUTPUT FOLDERS - one for each stage ======
folders = {
    "input":     "output_member2/1_input_frames",
    "gray":      "output_member2/2_gray_frames",
    "blur":      "output_member2/3_blur_frames",
    "blackhat":  "output_member2/4_blackhat_frames",
    "thresh":    "output_member2/5_thresh_frames",
    "opening":   "output_member2/6_opening_frames",
    "dilation":  "output_member2/7_dilation_frames",
    "edges":     "output_member2/8_edges_frames",
    "contours":  "output_member2/9_contours_frames",
}

# Create all folders
for folder in folders.values():
    os.makedirs(folder, exist_ok=True)

# ====== PROCESS FRAMES 17 TO 30 ======
for i in range(17, 31):

    # INPUT FRAME
    img_path = os.path.join(input_folder, f"gaussian_contrast_frame_{i}.jpg")
    image = cv2.imread(img_path)

    if image is None:
        print(f"Frame {i} not found, skipping...")
        continue

    # Save input
    cv2.imwrite(os.path.join(folders["input"], f"frame_{i}.jpg"), image)

    # PREPROCESSING - GRAYSCALE
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    cv2.imwrite(os.path.join(folders["gray"], f"frame_{i}.jpg"), gray)

    # PREPROCESSING - GAUSSIAN BLUR
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    cv2.imwrite(os.path.join(folders["blur"], f"frame_{i}.jpg"), blur)

    # SEGMENTATION - BLACKHAT
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    blackhat = cv2.morphologyEx(blur, cv2.MORPH_BLACKHAT, kernel)
    cv2.imwrite(os.path.join(folders["blackhat"], f"frame_{i}.jpg"), blackhat)

    # SEGMENTATION - THRESHOLD (Otsu for better accuracy)
    norm = cv2.normalize(blackhat, None, 0, 255, cv2.NORM_MINMAX)
    _, thresh = cv2.threshold(norm, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cv2.imwrite(os.path.join(folders["thresh"], f"frame_{i}.jpg"), thresh)

    # MORPHOLOGICAL CLEANING - OPENING
    kernel_small = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_small)
    cv2.imwrite(os.path.join(folders["opening"], f"frame_{i}.jpg"), opening)

    # MORPHOLOGICAL CLEANING - DILATION (used as predicted mask)
    dilation = cv2.dilate(opening, kernel_small, iterations=2)
    cv2.imwrite(os.path.join(folders["dilation"], f"frame_{i}.jpg"), dilation)

    # EDGE DETECTION
    edges = cv2.Canny(dilation, 50, 150)
    cv2.imwrite(os.path.join(folders["edges"], f"frame_{i}.jpg"), edges)

    # CONTOUR DETECTION + FILTERING
    contours, _ = cv2.findContours(dilation, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    output = image.copy()

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area > 200:
            x, y, w, h = cv2.boundingRect(cnt)
            roi = dilation[y:y+h, x:x+w]
            density = cv2.countNonZero(roi) / (w * h)

            if 0.15 < density < 0.5:
                cv2.rectangle(output, (x, y), (x+w, y+h), (0, 255, 0), 2)
                cv2.putText(output, "Alligator Crack", (x, y-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    cv2.imwrite(os.path.join(folders["contours"], f"frame_{i}.jpg"), output)

    print(f"Frame {i} processed and saved.")

print("\nAll frames processed successfully!")
print("\nFolder structure created:")
for name, path in folders.items():
    print(f"  {path}/")

Frame 17 processed and saved.
Frame 18 processed and saved.
Frame 19 processed and saved.
Frame 20 processed and saved.
Frame 21 processed and saved.
Frame 22 processed and saved.
Frame 23 processed and saved.
Frame 24 processed and saved.
Frame 25 processed and saved.
Frame 26 processed and saved.
Frame 27 processed and saved.
Frame 28 processed and saved.
Frame 29 processed and saved.
Frame 30 processed and saved.

All frames processed successfully!

Folder structure created:
  output_member2/1_input_frames/
  output_member2/2_gray_frames/
  output_member2/3_blur_frames/
  output_member2/4_blackhat_frames/
  output_member2/5_thresh_frames/
  output_member2/6_opening_frames/
  output_member2/7_dilation_frames/
  output_member2/8_edges_frames/
  output_member2/9_contours_frames/


In [3]:

# POTHOLE SEGMENTATION

import cv2
import os
import numpy as np
 
input_folder = "road2_member2_gaussian_contrast"
 
# Separate folder for each stage
folders_cell3 = {
    "input":     "segmentation_results1/1_input_frames",
    "gray":      "segmentation_results1/2_gray_frames",
    "blur":      "segmentation_results1/3_blur_frames",
    "mask":      "segmentation_results1/4_mask_frames",
    "contours":  "segmentation_results1/5_contour_frames",
    "detected":  "segmentation_results1/6_detected_frames",
}
 
# Create all folders
for folder in folders_cell3.values():
    os.makedirs(folder, exist_ok=True)
 
min_area = 200
max_area = 5000
 
for file in os.listdir(input_folder):
    path = os.path.join(input_folder, file)
    img = cv2.imread(path)
    if img is None:
        continue
 
    # Save input
    cv2.imwrite(os.path.join(folders_cell3["input"], file), img)
 
    # Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    cv2.imwrite(os.path.join(folders_cell3["gray"], file), gray)
 
    # Gaussian Blur
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    cv2.imwrite(os.path.join(folders_cell3["blur"], file), blur)
 
    # Threshold mask
    _, mask = cv2.threshold(blur, 120, 255, cv2.THRESH_BINARY_INV)
    cv2.imwrite(os.path.join(folders_cell3["mask"], file), mask)
 
    # Contour detection
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
 
    marked = img.copy()
    mask_valid = np.zeros_like(mask)
 
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if min_area < area < max_area:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(marked, (x, y), (x+w, y+h), (0, 0, 255), 2)
            cv2.putText(marked, "Pothole", (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            cv2.rectangle(mask_valid, (x, y), (x+w, y+h), 255, -1)
 
    # Save contour mask and detected output separately
    cv2.imwrite(os.path.join(folders_cell3["contours"], file), mask_valid)
    cv2.imwrite(os.path.join(folders_cell3["detected"], file), marked)
 
    print(f"Cell3 - {file} processed and saved.")
 
print("\nCell 3 completed!")
print("Folder structure:")
for name, path in folders_cell3.items():
    print(f"  {path}/")

Cell3 - frame_1.jpg processed and saved.
Cell3 - frame_10.jpg processed and saved.
Cell3 - frame_11.jpg processed and saved.
Cell3 - frame_12.jpg processed and saved.
Cell3 - frame_13.jpg processed and saved.
Cell3 - frame_15.jpg processed and saved.
Cell3 - frame_16.jpg processed and saved.
Cell3 - frame_17.jpg processed and saved.
Cell3 - frame_18.jpg processed and saved.
Cell3 - frame_19.jpg processed and saved.
Cell3 - frame_2.jpg processed and saved.
Cell3 - frame_20.jpg processed and saved.
Cell3 - frame_21.jpg processed and saved.
Cell3 - frame_22.jpg processed and saved.
Cell3 - frame_25.jpg processed and saved.
Cell3 - frame_27.jpg processed and saved.
Cell3 - frame_28.jpg processed and saved.
Cell3 - frame_29.jpg processed and saved.
Cell3 - frame_3.jpg processed and saved.
Cell3 - frame_30.jpg processed and saved.
Cell3 - frame_4.jpg processed and saved.
Cell3 - frame_5.jpg processed and saved.
Cell3 - frame_6.jpg processed and saved.
Cell3 - frame_7.jpg processed and saved.

In [4]:

# MANUAL GROUND TRUTH MASK DRAWING
# Draw on ORIGINAL COLOR IMAGE - filled area saved as mask
# Controls:
#   Left Click + Drag → Paint/Fill damage area (shown in RED)
#   S → Save mask and move to next frame
#   C → Clear current drawing
#   ESC → Skip this frame

 
import cv2
import numpy as np
import os
 
# Ground truth save folder
gt_folder = "ground_truth_masks"
os.makedirs(gt_folder, exist_ok=True)
 
# ---- 6 Selected frames (2 per damage type) ----
frames_to_annotate = [
    # (image path,                                           save name,      damage type)
    ("road_member2_gaussian_contrast/frame_1.jpg",           "mask_1.jpg",   "Crack"),
    ("road_member2_gaussian_contrast/frame_5.jpg",           "mask_5.jpg",   "Crack"),
    ("road_member3_sharpened/gaussian_contrast_frame_20.jpg","mask_20.jpg",  "Alligator Crack"),
    ("road_member3_sharpened/gaussian_contrast_frame_25.jpg","mask_25.jpg",  "Alligator Crack"),
    ("road2_member2_gaussian_contrast/frame_15.jpg",         "mask_p15.jpg", "Pothole"),
    ("road2_member2_gaussian_contrast/frame_19.jpg",         "mask_p19.jpg", "Pothole"),
]
 
brush_size = 20  # bigger brush for easier filling
 
# ---- Loop through each frame ----
for idx, (img_path, save_name, damage_type) in enumerate(frames_to_annotate):
 
    print(f"\n{'='*55}")
    print(f"  Frame {idx+1} of {len(frames_to_annotate)}")
    print(f"  Damage Type : {damage_type}")
    print(f"  Save as     : {save_name}")
    print(f"{'='*55}")
    print("  Controls:")
    print("  Left Click + Drag → Paint RED over damage area")
    print("  S                 → Save and go to next frame")
    print("  C                 → Clear and start again")
    print("  ESC               → Skip this frame")
    print("  + / -             → Increase / Decrease brush size")
 
    # Load original color image
    img = cv2.imread(img_path)
 
    if img is None:
        print(f"  ERROR: Image not found → {img_path}")
        continue
 
    # Binary mask - same size as image, all black at start
    mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
 
    drawing = False
    current_brush = [brush_size]  # use list so inner function can modify
 
    def draw(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            drawing_state[0] = True
        elif event == cv2.EVENT_MOUSEMOVE:
            if drawing_state[0]:
                # Fill circle on mask
                cv2.circle(mask, (x, y), current_brush[0], 255, -1)
        elif event == cv2.EVENT_LBUTTONUP:
            drawing_state[0] = False
 
    drawing_state = [False]
 
    # Window name
    win = f"Draw on Image - {damage_type} ({idx+1}/{len(frames_to_annotate)}) | S=Save C=Clear ESC=Skip +/-=Brush"
 
    cv2.namedWindow(win, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win, 600, 800)
    cv2.setMouseCallback(win, draw)
 
    saved = False
 
    while True:
        # Show original image with RED overlay on drawn area
        display = img.copy()
 
        # Apply red color where mask is drawn
        red_overlay = display.copy()
        red_overlay[mask == 255] = [0, 0, 255]  # red on damage area
 
        # Blend original and overlay so you can still see road underneath
        display = cv2.addWeighted(display, 0.5, red_overlay, 0.5, 0)
 
        # Draw brush size circle indicator at top left
        cv2.circle(display, (current_brush[0] + 10, current_brush[0] + 10),
                   current_brush[0], (0, 255, 255), 2)
        cv2.putText(display, f"Brush: {current_brush[0]}px", (10, current_brush[0]*2 + 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        cv2.putText(display, f"S=Save  C=Clear  ESC=Skip  +/-=Brush Size", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
 
        cv2.imshow(win, display)
 
        key = cv2.waitKey(1) & 0xFF
 
        # S - Save mask
        if key == ord('s'):
            save_path = os.path.join(gt_folder, save_name)
            cv2.imwrite(save_path, mask)
            print(f"\n  ✓ Saved: {save_path}")
            saved = True
            break
 
        # C - Clear mask
        elif key == ord('c'):
            mask[:] = 0
            print("  Mask cleared. Draw again.")
 
        # ESC - Skip
        elif key == 27:
            print(f"  Skipped: {img_path}")
            break
 
        # + increase brush size
        elif key == ord('+') or key == ord('='):
            current_brush[0] = min(current_brush[0] + 5, 80)
            print(f"  Brush size: {current_brush[0]}")
 
        # - decrease brush size
        elif key == ord('-'):
            current_brush[0] = max(current_brush[0] - 5, 5)
            print(f"  Brush size: {current_brush[0]}")
 
    cv2.destroyAllWindows()
 
    if saved:
        print(f"  Ground truth mask saved for {damage_type} → {save_name}")
    else:
        print(f"  No mask saved for {save_name}")
 
print("\n" + "="*55)
print("Ground truth annotation completed!")
print(f"All masks saved in: {gt_folder}/")
print("\nSaved masks:")
for f in sorted(os.listdir(gt_folder)):
    print(f"  {f}")


  Frame 1 of 6
  Damage Type : Crack
  Save as     : mask_1.jpg
  Controls:
  Left Click + Drag → Paint RED over damage area
  S                 → Save and go to next frame
  C                 → Clear and start again
  ESC               → Skip this frame
  + / -             → Increase / Decrease brush size
  Brush size: 15
  Brush size: 10
  Brush size: 5

  ✓ Saved: ground_truth_masks\mask_1.jpg
  Ground truth mask saved for Crack → mask_1.jpg

  Frame 2 of 6
  Damage Type : Crack
  Save as     : mask_5.jpg
  Controls:
  Left Click + Drag → Paint RED over damage area
  S                 → Save and go to next frame
  C                 → Clear and start again
  ESC               → Skip this frame
  + / -             → Increase / Decrease brush size
  Brush size: 15
  Brush size: 10
  Brush size: 5

  ✓ Saved: ground_truth_masks\mask_5.jpg
  Ground truth mask saved for Crack → mask_5.jpg

  Frame 3 of 6
  Damage Type : Alligator Crack
  Save as     : mask_20.jpg
  Controls:
  Left Click

In [5]:
import os
os.makedirs("predicted_masks", exist_ok=True)

In [6]:

# AUTO COPY PREDICTED MASKS FOR SAME 6 FRAMES
# Copies matching predicted masks from segmentation outputs

 
import shutil
import os
 
pred_folder = "predicted_masks"
os.makedirs(pred_folder, exist_ok=True)
 
# Match predicted masks to ground truth frames
# (predicted source path,                                          saved name)
predicted_masks = [
    ("segmentation_results/4_morph_frames/frame_1.jpg",            "mask_1.jpg"),
    ("segmentation_results/4_morph_frames/frame_5.jpg",            "mask_5.jpg"),
    ("output_member2/7_dilation_frames/frame_20.jpg",              "mask_20.jpg"),
    ("output_member2/7_dilation_frames/frame_25.jpg",              "mask_25.jpg"),
    ("segmentation_results1/5_contour_frames/frame_15.jpg",        "mask_p15.jpg"),
    ("segmentation_results1/5_contour_frames/frame_19.jpg",        "mask_p19.jpg"),
]
 
print("Copying predicted masks...")
for src, save_name in predicted_masks:
    dst = os.path.join(pred_folder, save_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied: {save_name}")
    else:
        print(f"Missing predicted mask: {src}")
 
print(f"\nAll predicted masks saved in: {pred_folder}/")

Copying predicted masks...
Copied: mask_1.jpg
Copied: mask_5.jpg
Copied: mask_20.jpg
Copied: mask_25.jpg
Copied: mask_p15.jpg
Copied: mask_p19.jpg

All predicted masks saved in: predicted_masks/


In [7]:
# METRICS CALCULATION FOR 6 FRAMES
# Compares ground truth vs predicted masks
# Generates Table A and Table B for report

 
import cv2
import numpy as np
import os
import csv
 
gt_folder   = "ground_truth_masks"
pred_folder = "predicted_masks"
 
# Frame info mapping
frame_info = {
    "mask_1.jpg":   ("Crack",          "Video 1"),
    "mask_5.jpg":   ("Crack",          "Video 1"),
    "mask_20.jpg":  ("Alligator Crack","Video 2"),
    "mask_25.jpg":  ("Alligator Crack","Video 2"),
    "mask_p15.jpg": ("Pothole",        "Video 3"),
    "mask_p19.jpg": ("Pothole",        "Video 3"),
}
 
def calculate_metrics(gt_path, pred_path):
    gt   = cv2.imread(gt_path,   cv2.IMREAD_GRAYSCALE)
    pred = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
 
    if gt is None or pred is None:
        print(f"Missing: {gt_path} or {pred_path}")
        return None
 
    if gt.shape != pred.shape:
        pred = cv2.resize(pred, (gt.shape[1], gt.shape[0]))
 
    gt   = (gt   > 127).astype(np.uint8)
    pred = (pred > 127).astype(np.uint8)
 
    TP = np.sum((gt == 1) & (pred == 1))
    TN = np.sum((gt == 0) & (pred == 0))
    FP = np.sum((gt == 0) & (pred == 1))
    FN = np.sum((gt == 1) & (pred == 0))
 
    accuracy    = (TP + TN)                / (TP + TN + FP + FN + 1e-6)
    precision   = TP                       / (TP + FP + 1e-6)
    recall      = TP                       / (TP + FN + 1e-6)
    f1          = (2 * precision * recall) / (precision + recall + 1e-6)
    iou         = TP                       / (TP + FP + FN + 1e-6)
    dice        = (2 * TP)                 / (2 * TP + FP + FN + 1e-6)
    specificity = TN                       / (TN + FP + 1e-6)
 
    return accuracy, precision, recall, f1, iou, dice, specificity
 
# ---- Calculate metrics for all 6 frames ----
all_results = []
 
for filename, (deg_type, video_id) in frame_info.items():
    gt_path   = os.path.join(gt_folder,   filename)
    pred_path = os.path.join(pred_folder, filename)
 
    if not os.path.exists(gt_path):
        print(f"Ground truth not found: {filename}, skipping...")
        continue
 
    if not os.path.exists(pred_path):
        print(f"Predicted mask not found: {filename}, skipping...")
        continue
 
    metrics = calculate_metrics(gt_path, pred_path)
 
    if metrics is not None:
        accuracy, precision, recall, f1, iou, dice, specificity = metrics
        all_results.append({
            "filename":    filename,
            "video_id":    video_id,
            "degradation": deg_type,
            "accuracy":    round(accuracy,    3),
            "precision":   round(precision,   3),
            "recall":      round(recall,      3),
            "f1":          round(f1,          3),
            "iou":         round(iou,         3),
            "dice":        round(dice,        3),
            "specificity": round(specificity, 3),
        })
 
# ============================================================
# TABLE A - Overall Quantitative Performance by Degradation Type
# ============================================================
 
print("\n" + "="*80)
print("TABLE A - Overall Quantitative Performance")
print("="*80)
print(f"{'Degradation Type':<20}{'Frames':>7}{'Accuracy':>10}{'Precision':>10}{'Recall':>8}{'F1':>7}{'IoU':>7}  Remarks")
print("-"*80)
 
degradation_groups = {}
for r in all_results:
    d = r["degradation"]
    if d not in degradation_groups:
        degradation_groups[d] = []
    degradation_groups[d].append(r)
 
table_a_rows = []
for deg, rows in degradation_groups.items():
    n        = len(rows)
    avg_acc  = round(np.mean([r["accuracy"]   for r in rows]), 3)
    avg_prec = round(np.mean([r["precision"]  for r in rows]), 3)
    avg_rec  = round(np.mean([r["recall"]     for r in rows]), 3)
    avg_f1   = round(np.mean([r["f1"]         for r in rows]), 3)
    avg_iou  = round(np.mean([r["iou"]        for r in rows]), 3)
    remark   = "Good" if avg_iou >= 0.7 else "Moderate" if avg_iou >= 0.5 else "Needs improvement"
 
    print(f"{deg:<20}{n:>7}{avg_acc:>10}{avg_prec:>10}{avg_rec:>8}{avg_f1:>7}{avg_iou:>7}  {remark}")
    table_a_rows.append([deg, n, avg_acc, avg_prec, avg_rec, avg_f1, avg_iou, remark])
 
# ============================================================
# TABLE B - Video-wise Performance
# ============================================================
 
print("\n" + "="*80)
print("TABLE B - Video-wise Performance")
print("="*80)
print(f"{'Video ID':<10}{'Degradation':<22}{'Frames':>7}{'Best IoU':>9}{'Avg IoU':>9}{'Avg Prec':>10}{'Avg Recall':>11}")
print("-"*80)
 
video_groups = {}
for r in all_results:
    v = r["video_id"]
    if v not in video_groups:
        video_groups[v] = []
    video_groups[v].append(r)
 
table_b_rows = []
for vid, rows in sorted(video_groups.items()):
    deg      = rows[0]["degradation"]
    n        = len(rows)
    best_iou = round(max( [r["iou"]       for r in rows]), 3)
    avg_iou  = round(np.mean([r["iou"]    for r in rows]), 3)
    avg_prec = round(np.mean([r["precision"] for r in rows]), 3)
    avg_rec  = round(np.mean([r["recall"] for r in rows]), 3)
 
    print(f"{vid:<10}{deg:<22}{n:>7}{best_iou:>9}{avg_iou:>9}{avg_prec:>10}{avg_rec:>11}")
    table_b_rows.append([vid, deg, n, best_iou, avg_iou, avg_prec, avg_rec])
 
# ============================================================
# SAVE TO CSV
# ============================================================
 
with open("quantitative_results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Frame", "Video ID", "Degradation", "Accuracy",
                     "Precision", "Recall", "F1-score", "IoU", "Dice", "Specificity"])
    for r in all_results:
        writer.writerow([r["filename"], r["video_id"], r["degradation"],
                         r["accuracy"], r["precision"], r["recall"],
                         r["f1"], r["iou"], r["dice"], r["specificity"]])
 
with open("table_a_overall.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Degradation Type", "No. of Test Frames", "Accuracy",
                     "Precision", "Recall", "F1-score", "IoU", "Remarks"])
    writer.writerows(table_a_rows)
 
with open("table_b_videowise.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Video ID", "Degradation Type", "Tested Frames",
                     "Best IoU", "Average IoU", "Average Precision", "Average Recall"])
    writer.writerows(table_b_rows)
 
print("\nCSV files saved:")
print("  quantitative_results.csv  → all 6 frames results")
print("  table_a_overall.csv       → Table A for report")
print("  table_b_videowise.csv     → Table B for report")


TABLE A - Overall Quantitative Performance
Degradation Type     Frames  Accuracy Precision  Recall     F1    IoU  Remarks
--------------------------------------------------------------------------------
Crack                     2     0.771     0.036   0.462  0.067  0.035  Needs improvement
Alligator Crack           2     0.597     0.166   0.758  0.272  0.158  Needs improvement
Pothole                   2     0.982       0.6   0.622  0.606  0.443  Needs improvement

TABLE B - Video-wise Performance
Video ID  Degradation            Frames Best IoU  Avg IoU  Avg Prec Avg Recall
--------------------------------------------------------------------------------
Video 1   Crack                       2    0.049    0.035     0.036      0.462
Video 2   Alligator Crack             2     0.18    0.158     0.166      0.758
Video 3   Pothole                     2    0.546    0.443       0.6      0.622

CSV files saved:
  quantitative_results.csv  → all 6 frames results
  table_a_overall.csv       →

In [8]:
import os
folder = "segmentation_results1/5_contour_frames"
print(os.listdir(folder))

['frame_1.jpg', 'frame_10.jpg', 'frame_11.jpg', 'frame_12.jpg', 'frame_13.jpg', 'frame_15.jpg', 'frame_16.jpg', 'frame_17.jpg', 'frame_18.jpg', 'frame_19.jpg', 'frame_2.jpg', 'frame_20.jpg', 'frame_21.jpg', 'frame_22.jpg', 'frame_25.jpg', 'frame_27.jpg', 'frame_28.jpg', 'frame_29.jpg', 'frame_3.jpg', 'frame_30.jpg', 'frame_4.jpg', 'frame_5.jpg', 'frame_6.jpg', 'frame_7.jpg', 'frame_8.jpg', 'frame_9.jpg']
